# 🧠 Train T5 — Note Structuring Model
**JSS SMC MCA Institute | Udaya Kumar | USN: P02ME24S126024**

Trains T5-small on CNN/DailyMail summarization dataset.
Input: raw lecture transcript → Output: structured academic notes.

**Runtime: GPU (T4) | ~2-3 hours for 3 epochs**


In [ ]:
# Cell 1 — Install & mount Drive
!pip install -q transformers datasets rouge-score evaluate

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/audio_project/models/t5-notes', exist_ok=True)
print('✅ Ready!')

In [ ]:
# Cell 2 — Load CNN/DailyMail dataset
from datasets import load_dataset

dataset = load_dataset('abisee/cnn_dailymail', '3.0.0')
print('Dataset loaded:')
print(dataset)
print('\nSample article (first 300 chars):')
print(dataset['train'][0]['article'][:300])
print('\nSample highlights:')
print(dataset['train'][0]['highlights'])

In [ ]:
# Cell 3 — Tokenize for T5
from transformers import T5Tokenizer

MODEL_NAME  = 'google-t5/t5-small'
tokenizer   = T5Tokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT   = 512
MAX_TARGET  = 128

def preprocess(batch):
    inputs = ['summarize: ' + doc for doc in batch['article']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT,
                             truncation=True, padding='max_length')
    labels = tokenizer(text_target=batch['highlights'],
                       max_length=MAX_TARGET,
                       truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Use 50k training samples (full = 300k, too slow)
train_small = dataset['train'].select(range(50000))
eval_small  = dataset['validation'].select(range(2000))

print('Tokenizing training data...')
tok_train = train_small.map(preprocess, batched=True,
    remove_columns=dataset['train'].column_names)

print('Tokenizing eval data...')
tok_eval  = eval_small.map(preprocess, batched=True,
    remove_columns=dataset['validation'].column_names)

print('✅ Tokenization complete!')

In [ ]:
# Cell 4 — Train T5
from transformers import (
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

model        = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge         = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels         = [[l for l in label if l != -100] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {'rouge1': result['rouge1'], 'rougeL': result['rougeL']}

T5_OUTPUT = '/content/drive/MyDrive/audio_project/models/t5-notes'

args = Seq2SeqTrainingArguments(
    output_dir=T5_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tok_train, eval_dataset=tok_eval,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics
)

print('🚀 Starting T5 training (~2-3 hrs)...')
trainer.train()
print('✅ T5 training done! Saved to:', T5_OUTPUT)

In [ ]:
# Cell 5 — Zip & download
import shutil
from google.colab import files

shutil.make_archive('/content/t5_notes', 'zip',
                    '/content/drive/MyDrive/audio_project/models/t5-notes')
files.download('/content/t5_notes.zip')
print('✅ Extract to: audio_notes_project/models/t5-notes/')